
# Parkinson's Disease – UPDRS Regression & Classification Notebook

This notebook provides a **clean, end-to-end pipeline** to:
1. Load the Parkinson’s dataset
2. Explore and prepare the data
3. **Regression**: Predict **`total_UPDRS`**
4. **Classification**: Convert `total_UPDRS` into a binary label and evaluate models (Accuracy, Precision, Recall, F1, ROC)

> **Note:** Set your dataset path in the **Configuration** cell below. By default, it looks for `data/parkinsons.csv`.


In [ ]:

# === Imports ===
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Regression models
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

# Classification models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Metrics
from sklearn.metrics import (
    mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, roc_auc_score, classification_report
)

import matplotlib.pyplot as plt

# === Configuration ===
DATA_PATH = 'data/parkinsons.csv'  # <- Change this if needed
ID_COLS = ['subject#']             # columns considered as IDs to drop from features
REG_TARGET = 'total_UPDRS'         # target for regression
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

print('Configured DATA_PATH =', DATA_PATH)


In [ ]:

# === Load Data ===
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Please place your CSV there or update DATA_PATH.\n"
        "Example: UCI Parkinson's dataset with UPDRS columns (e.g., total_UPDRS, motor_UPDRS)."
    )

df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())
print('\nColumns:', list(df.columns))


## Quick EDA

In [ ]:

display(df.describe(include='all'))
na_counts = df.isna().sum()
print('Missing values per column:\n', na_counts[na_counts>0])


## Regression: Predict `total_UPDRS`

In [ ]:

# === Prepare features/target for regression ===
if REG_TARGET not in df.columns:
    raise KeyError(f"'{REG_TARGET}' not found in columns. Available: {list(df.columns)}")

X_reg = df.drop(columns=[REG_TARGET] + [c for c in ID_COLS if c in df.columns], errors='ignore')
y_reg = df[REG_TARGET].values

print('Features shape:', X_reg.shape, '| Target shape:', y_reg.shape)


In [ ]:

# === Build pipelines ===
reg_models = {
    'SVR': Pipeline([('scaler', StandardScaler()), ('model', SVR())]),
    'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor())]),
    'Lasso': Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.001, max_iter=5000))]),
    'RandomForest': Pipeline([('model', RandomForestRegressor(random_state=42))])
}

# === Cross-validation MSE ===
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, pipe in reg_models.items():
    scores = cross_val_score(pipe, X_reg, y_reg, cv=cv, scoring='neg_mean_squared_error')
    mse_scores = -scores
    cv_results[name] = mse_scores
    print(f"{name} CV MSE: {mse_scores.mean():.4f} ± {mse_scores.std():.4f}")


In [ ]:

# === Train/Test Split (Regression) ===
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

reg_test_results = {}
for name, pipe in reg_models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    reg_test_results[name] = mse
    print(f"{name} Test MSE: {mse:.4f}")


## Classification: Convert `total_UPDRS` to Binary Target

In [ ]:

# === Define binary target from total_UPDRS ===
threshold = df[REG_TARGET].median()  # balanced split by default
print('Classification threshold (median):', threshold)

df['target'] = (df[REG_TARGET] > threshold).astype(int)
print(df['target'].value_counts())


In [ ]:

# === Prepare features/target for classification ===
X_cls = df.drop(columns=['target', REG_TARGET] + [c for c in ID_COLS if c in df.columns], errors='ignore')
y_cls = df['target'].values

Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

cls_models = {
    'LogisticRegression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000))]),
    'RandomForest': Pipeline([('model', RandomForestClassifier(random_state=42))])
}

def evaluate_classifier(y_true, y_pred, name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n--- {name} ---")
    print('Accuracy :', f"{acc:.4f}")
    print('Precision:', f"{prec:.4f}")
    print('Recall   :', f"{rec:.4f}")
    print('F1       :', f"{f1:.4f}")
    print('Confusion Matrix:\n', confusion_matrix(y_true, y_pred))
    print('\nClassification report:\n', classification_report(y_true, y_pred, zero_division=0))
    return acc, prec, rec, f1

cls_results = {}
for name, pipe in cls_models.items():
    pipe.fit(Xc_train, yc_train)
    y_pred = pipe.predict(Xc_test)
    cls_results[name] = evaluate_classifier(yc_test, y_pred, name)


In [ ]:

# === ROC Curves ===
plt.figure(figsize=(6,4))

for name, pipe in cls_models.items():
    # predict_proba may not exist for some models; guard it
    proba_method = getattr(pipe, 'predict_proba', None)
    if proba_method is None:
        # try decision_function if available
        decision_method = getattr(pipe, 'decision_function', None)
        if decision_method is None:
            continue
        scores = pipe.decision_function(Xc_test)
        # scale to 0..1 via min-max for ROC AUC (not ideal, but a fallback)
        scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
        y_scores = scores
    else:
        y_scores = pipe.predict_proba(Xc_test)[:,1]
    fpr, tpr, _ = roc_curve(yc_test, y_scores)
    auc = roc_auc_score(yc_test, y_scores)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc:.3f}")

plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'roc_curve.png'), dpi=150)
plt.show()

print(f"ROC curve saved to {os.path.join(RESULTS_DIR, 'roc_curve.png')}")


## Summary Tables for README

In [ ]:

# Build concise tables to copy into README

# Regression (Test MSE)
reg_table = pd.DataFrame(
    {'Model': list(reg_test_results.keys()),
     'Test_MSE': [round(v, 4) for v in reg_test_results.values()]}
).sort_values('Test_MSE')
display(reg_table)

# Classification metrics
cls_table = pd.DataFrame(
    [{'Model': m,
      'Accuracy': round(v[0],4),
      'Precision': round(v[1],4),
      'Recall': round(v[2],4),
      'F1': round(v[3],4)} for m, v in cls_results.items()]
).sort_values('F1', ascending=False)
display(cls_table)

# Save to CSV for convenience
reg_table.to_csv(os.path.join(RESULTS_DIR, 'regression_test_mse.csv'), index=False)
cls_table.to_csv(os.path.join(RESULTS_DIR, 'classification_metrics.csv'), index=False)

print('Saved:')
print(' -', os.path.join(RESULTS_DIR, 'regression_test_mse.csv'))
print(' -', os.path.join(RESULTS_DIR, 'classification_metrics.csv'))
